In [2]:
import torch
import gpytorch
import pandas as pd
import pickle
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from torch.utils.data import TensorDataset, DataLoader
from pyproj import Transformer
from sklearn.metrics import pairwise_distances
from scipy.interpolate import RegularGridInterpolator
from torch_geometric.data import Data




/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pandas as pd

air_korea_final = pd.read_pickle("/Users/drewbaldwin/PM2_5 Research/air_korea_final_imputed.pkl")
air_korea_final.head()


,Datetime,SO2,CO,O3,NO2,PM10,PM25,Station_ID,Year,lon,...,major_roads_count_3km,total_road_length_3km,urban_landuse_area_m2_3km,green_space_area_3km,building_footprint_area_3km,railway_length_3km,dist_to_coast_km,dist_to_major_road_km,industrial_area_m2_3km,traffic_points_count_3km
0,2016-01-01 00:00:00,7.0,1000.0,2.0,76.0,77.0,53.0,111121,2016,126.9747,...,613.0,923909.036845,2.952433e+06,9.417996e+06,4.942823e+06,114737.280122,10.830496,0.102038,1489.332477,159.0
1,2016-01-01 01:00:00,7.0,1100.0,2.0,77.0,70.0,48.0,111121,2016,126.9747,...,613.0,923909.036845,2.952433e+06,9.417996e+06,4.942823e+06,114737.280122,10.830496,0.102038,1489.332477,159.0
2,2016-01-01 02:00:00,7.0,1200.0,2.0,78.0,75.0,53.0,111121,2016,126.9747,...,613.0,923909.036845,2.952433e+06,9.417996e+06,4.942823e+06,114737.280122,10.830496,0.102038,1489.332477,159.0
3,2016-01-01 03:00:00,6.0,1400.0,2.0,78.0,77.0,53.0,111121,2016,126.9747,...,613.0,923909.036845,2.952433e+06,9.417996e+06,4.942823e+06,114737.280122,10.830496,0.102038,1489.332477,159.0
4,2016-01-01 04:00:00,6.0,1500.0,2.0,77.0,83.0,52.0,111121,2016,126.9747,...,613.0,923909.036845,2.952433e+06,9.417996e+06,4.942823e+06,114737.280122,10.830496,0.102038,1489.332477,159.0


In [ ]:
#read in the networks and features


In [13]:

BASE = "/Users/drewbaldwin/PM2_5 Research"

# ---------------- Node features (weather anomalies, static z-scores, PM25 anomaly target) ----------------
features = pd.read_pickle(f"{BASE}/model_features.pkl")
print(f"features: {features.shape}")
print(features.columns.tolist())

static_scaler = pd.read_pickle(f"{BASE}/static_feature_scaler.pkl")
pm25_scaler = pd.read_pickle(f"{BASE}/pm25_anomaly_scaler.pkl")
print(f"\nstatic_scaler: {static_scaler.shape}, pm25_scaler: {pm25_scaler.shape}")

# ---------------- Graph structure: candidate edges + hourly wind ----------------
candidate_edges = pd.read_pickle(f"{BASE}/candidate_edges.pkl")
print(f"\ncandidate_edges: {candidate_edges.shape}")

wind_npz = np.load(f"{BASE}/hourly_wind_wide.npz", allow_pickle=True)
wind_speed_wide = wind_npz["wind_speed"]
wind_bearing_wide = wind_npz["wind_bearing"]
station_order = list(wind_npz["station_order"])
station_index = {sid: i for i, sid in enumerate(station_order)}
hourly_index = pd.to_datetime(wind_npz["hourly_index"])
n_stations = len(station_order)
print(f"wind arrays: {wind_speed_wide.shape} (hours x stations), {n_stations} stations, "
      f"{len(hourly_index):,} hours ({hourly_index[0]} to {hourly_index[-1]})")

WIND_WEIGHT = 0.15
nonzero_speeds = wind_speed_wide[wind_speed_wide > 0.1]
REFERENCE_SPEED = float(np.percentile(nonzero_speeds, 75))
print(f"REFERENCE_SPEED: {REFERENCE_SPEED:.2f}")

def circular_diff(a, b):
    d = np.abs(a - b) % 360
    return np.minimum(d, 360 - d)

def edge_weights_at(hour_idx: int) -> pd.DataFrame:
    """Directed edge weights for a single hour (by position in hourly_index)."""
    i_idx = candidate_edges["i_idx"].to_numpy()
    wind_bearing_i = wind_bearing_wide[hour_idx, i_idx]
    wind_speed_i = wind_speed_wide[hour_idx, i_idx]

    angle_diff = circular_diff(wind_bearing_i, candidate_edges["bearing_ij"].to_numpy())
    alignment_signed = np.cos(np.radians(angle_diff))
    speed_factor = np.minimum(wind_speed_i / REFERENCE_SPEED, 1.0)

    weight = candidate_edges["distance_decay"].to_numpy() * (1 + WIND_WEIGHT * alignment_signed * speed_factor)

    out = candidate_edges[["Station_i", "Station_j", "i_idx", "j_idx"]].copy()
    out["weight"] = weight
    return out

# ---------------- Sanity check everything lines up ----------------
example = edge_weights_at(0)
print(f"\nExample hour ({hourly_index[0]}): {len(example)} directed edges, "
      f"weight range [{example['weight'].min():.3f}, {example['weight'].max():.3f}]")

feature_stations = set(features["Station_ID"].unique())
graph_stations = set(station_order)
assert feature_stations == graph_stations, (
    f"station mismatch between features and graph: "
    f"{len(feature_stations - graph_stations)} only in features, "
    f"{len(graph_stations - feature_stations)} only in graph"
)
feature_hours = set(features["Datetime"].unique())
graph_hours = set(hourly_index)
assert feature_hours == graph_hours, "hour mismatch between features and graph"
print("\nStations and hours match exactly between features and graph -- ready to build the model.")


features: (9259008, 23)
['Datetime', 'Station_ID', 'temperature_2m_anomaly', 'relative_humidity_2m_anomaly', 'surface_pressure_anomaly', 'windspeed_10m_anomaly', 'temperature_2m_hourly_mean', 'relative_humidity_2m_hourly_mean', 'surface_pressure_hourly_mean', 'windspeed_10m_hourly_mean', 'precipitation_feature', 'elevation_m_z', 'major_roads_count_3km_z', 'total_road_length_3km_z', 'urban_landuse_area_m2_3km_z', 'green_space_area_3km_z', 'building_footprint_area_3km_z', 'railway_length_3km_z', 'dist_to_coast_km_z', 'dist_to_major_road_km_z', 'industrial_area_m2_3km_z', 'traffic_points_count_3km_z', 'PM25_anomaly']

static_scaler: (11, 2), pm25_scaler: (52608, 3)

candidate_edges: (5874, 7)
wind arrays: (52608, 176) (hours x stations), 176 stations, 52,608 hours (2016-01-01 00:00:00 to 2021-12-31 23:00:00)
REFERENCE_SPEED: 13.20

Example hour (2016-01-01 00:00:00): 5874 directed edges, weight range [0.327, 1.054]

Stations and hours match exactly between features and graph -- ready to b

In [14]:
#model 
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

FEATURE_COLS = [
    "temperature_2m_anomaly", "relative_humidity_2m_anomaly", "surface_pressure_anomaly", "windspeed_10m_anomaly",
    "temperature_2m_hourly_mean", "relative_humidity_2m_hourly_mean", "surface_pressure_hourly_mean", "windspeed_10m_hourly_mean",
    "precipitation_feature",
    "elevation_m_z", "major_roads_count_3km_z", "total_road_length_3km_z", "urban_landuse_area_m2_3km_z",
    "green_space_area_3km_z", "building_footprint_area_3km_z", "railway_length_3km_z",
    "dist_to_coast_km_z", "dist_to_major_road_km_z", "industrial_area_m2_3km_z", "traffic_points_count_3km_z",
    "PM25_anomaly",  # past target values included as an autoregressive input feature
]
N_FEATURES = len(FEATURE_COLS)
TARGET_COL = "PM25_anomaly"

n_hours = len(hourly_index)
X = np.zeros((n_hours, n_stations, N_FEATURES), dtype="float32")
for f_idx, col in enumerate(FEATURE_COLS):
    wide = features.pivot(index="Datetime", columns="Station_ID", values=col)[station_order]
    X[:, :, f_idx] = wide.to_numpy()

y = features.pivot(index="Datetime", columns="Station_ID", values=TARGET_COL)[station_order].to_numpy().astype("float32")

print(f"X: {X.shape} (hours, stations, features)")
print(f"y: {y.shape} (hours, stations)")
assert not np.isnan(X).any() and not np.isnan(y).any(), "unexpected NaNs in tensors"


X: (52608, 176, 21) (hours, stations, features)
y: (52608, 176) (hours, stations)


In [15]:
WINDOW = 24   # hours of input history
HORIZON = 1   # predict this many hours ahead

years = hourly_index.year
train_mask = years <= 2019
val_mask = years == 2020
test_mask = years == 2021

def valid_start_indices(mask):
    idx = np.where(mask)[0]
    # a window starting at t needs [t, t+WINDOW) for input and t+WINDOW+HORIZON-1 for target,
    # all within the same split (no windows straddling a split boundary)
    valid = [t for t in idx if t + WINDOW + HORIZON - 1 <= idx.max() and mask[t:t + WINDOW + HORIZON].all()]
    return np.array(valid)

train_starts = valid_start_indices(train_mask)
val_starts = valid_start_indices(val_mask)
test_starts = valid_start_indices(test_mask)
print(f"train windows: {len(train_starts):,}, val: {len(val_starts):,}, test: {len(test_starts):,}")


train windows: 35,040, val: 8,760, test: 8,736


In [21]:
def circular_diff(a, b):
    d = np.abs(a - b) % 360
    return np.minimum(d, 360 - d)

def edge_weights_batch(hour_indices: np.ndarray) -> np.ndarray:
    """Edge weights for an array of hours at once -> shape (len(hour_indices), n_edges)."""
    i_idx = candidate_edges["i_idx"].to_numpy()
    bearing_ij = candidate_edges["bearing_ij"].to_numpy()
    distance_decay = candidate_edges["distance_decay"].to_numpy()

    wind_bearing_i = wind_bearing_wide[np.ix_(hour_indices, i_idx)]   # (T, E)
    wind_speed_i = wind_speed_wide[np.ix_(hour_indices, i_idx)]       # (T, E)

    angle_diff = circular_diff(wind_bearing_i, bearing_ij[None, :])
    alignment_signed = np.cos(np.radians(angle_diff))
    speed_factor = np.minimum(wind_speed_i / REFERENCE_SPEED, 1.0)

    weight = distance_decay[None, :] * (1 + WIND_WEIGHT * alignment_signed * speed_factor)
    return weight.astype("float32")

src_idx = torch.tensor(candidate_edges["i_idx"].to_numpy(), dtype=torch.long)
dst_idx = torch.tensor(candidate_edges["j_idx"].to_numpy(), dtype=torch.long)

class WeightedGraphConv(nn.Module):
    """h_j = ReLU(W_self @ x_j + W_neigh @ weighted_mean_i(weight(i->j) * x_i))"""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.self_lin = nn.Linear(in_dim, out_dim)
        self.neigh_lin = nn.Linear(in_dim, out_dim)

    def forward(self, x, edge_weight):
        messages = x[:, src_idx, :] * edge_weight.unsqueeze(-1)          # (B, n_edges, in_dim)
        aggregated = torch.zeros_like(x)
        aggregated.index_add_(1, dst_idx, messages)                      # weighted SUM

        weight_sum = torch.zeros(x.shape[0], x.shape[1], 1, device=x.device)
        weight_sum.index_add_(1, dst_idx, edge_weight.unsqueeze(-1))     # total incoming weight per node
        aggregated = aggregated / (weight_sum + 1e-6)                    # -> weighted MEAN, not sum

        return torch.relu(self.self_lin(x) + self.neigh_lin(aggregated))



In [22]:
class SpatioTemporalGNN(nn.Module):
    def __init__(self, n_features, gnn_hidden=32, gru_hidden=32):
        super().__init__()
        self.gcn = WeightedGraphConv(n_features, gnn_hidden)
        self.gru = nn.GRU(gnn_hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_weight_window):
        # x_window: (B, W, n_stations, n_features), edge_weight_window: (B, W, n_edges)
        B, W, S, F = x_window.shape
        embeddings = []
        for t in range(W):
            embeddings.append(self.gcn(x_window[:, t], edge_weight_window[:, t]))
        h_seq = torch.stack(embeddings, dim=1)                    # (B, W, S, gnn_hidden)
        h_seq = h_seq.permute(0, 2, 1, 3).reshape(B * S, W, -1)   # (B*S, W, gnn_hidden)
        _, h_last = self.gru(h_seq)                                # (1, B*S, gru_hidden)
        pred = self.head(h_last.squeeze(0)).view(B, S)             # (B, S)
        return pred

class NoGraphGRU(nn.Module):
    """Same temporal setup, no spatial mixing -- isolates whether the graph adds value."""
    def __init__(self, n_features, hidden=32):
        super().__init__()
        self.gru = nn.GRU(n_features, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x_window, edge_weight_window=None):
        B, W, S, F = x_window.shape
        x_seq = x_window.permute(0, 2, 1, 3).reshape(B * S, W, F)
        _, h_last = self.gru(x_seq)
        return self.head(h_last.squeeze(0)).view(B, S)


In [23]:
def make_batch(starts):
    x_batch = np.stack([X[t:t + WINDOW] for t in starts])                          # (B, W, S, F)
    y_batch = np.stack([y[t + WINDOW + HORIZON - 1] for t in starts])              # (B, S)
    hour_idx_batch = np.stack([np.arange(t, t + WINDOW) for t in starts])          # (B, W)
    edge_w_batch = np.stack([edge_weights_batch(h) for h in hour_idx_batch])       # (B, W, n_edges)
    return (
        torch.tensor(x_batch), torch.tensor(y_batch),
        torch.tensor(edge_w_batch),
    )

def evaluate(model, starts, batch_size=128):
    model.eval() if model is not None else None
    preds, trues = [], []
    with torch.no_grad():
        for i in range(0, len(starts), batch_size):
            xb, yb, ewb = make_batch(starts[i:i + batch_size])
            if model is None:
                continue
            pred = model(xb, ewb)
            preds.append(pred.numpy())
            trues.append(yb.numpy())
    return np.concatenate(preds), np.concatenate(trues)

def train_model(model, train_starts, val_starts, epochs=5, batch_size=64, samples_per_epoch=3000, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    for epoch in range(epochs):
        model.train()
        epoch_starts = np.random.choice(train_starts, size=samples_per_epoch, replace=False)
        total_loss = 0.0
        for i in range(0, samples_per_epoch, batch_size):
            batch_starts = epoch_starts[i:i + batch_size]
            xb, yb, ewb = make_batch(batch_starts)
            opt.zero_grad()
            pred = model(xb, ewb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()
            total_loss += loss.item() * len(batch_starts)
        val_preds, val_trues = evaluate(model, val_starts[:2000])
        val_mse = np.mean((val_preds - val_trues) ** 2)
        print(f"epoch {epoch+1}: train MSE {total_loss/samples_per_epoch:.4f}, val MSE {val_mse:.4f}")
    return model

def mse_mae(preds, trues):
    return np.mean((preds - trues) ** 2), np.mean(np.abs(preds - trues))


In [ ]:
#run everything 
np.random.seed(0)
torch.manual_seed(0)

print("Training SpatioTemporalGNN (GCN + GRU)...")
gnn_model = SpatioTemporalGNN(N_FEATURES)
gnn_model = train_model(gnn_model, train_starts, val_starts,epochs=20,samples_per_epoch=5000)

print("\nTraining NoGraphGRU (ablation)...")
nograph_model = NoGraphGRU(N_FEATURES)
nograph_model = train_model(nograph_model, train_starts, val_starts)

# ---------------- Evaluate everything on the test set ----------------
test_subset = test_starts[:3000]  # cap for speed on CPU; increase once this all works

gnn_preds, trues = evaluate(gnn_model, test_subset)
nograph_preds, _ = evaluate(nograph_model, test_subset)
zero_preds = np.zeros_like(trues)
persistence_preds = np.stack([X[t + WINDOW - 1, :, FEATURE_COLS.index("PM25_anomaly")] for t in test_subset])

results = {}
for name, preds in [
    ("Zero baseline", zero_preds),
    ("Persistence baseline", persistence_preds),
    ("No-graph GRU", nograph_preds),
    ("GCN + GRU", gnn_preds),
]:
    mse, mae = mse_mae(preds, trues)
    results[name] = (mse, mae)

print(f"\n{'Model':<22} {'MSE (anomaly)':>15} {'MAE (anomaly)':>15}")
for name, (mse, mae) in results.items():
    print(f"{name:<22} {mse:>15.4f} {mae:>15.4f}")


Training SpatioTemporalGNN (GCN + GRU)...
epoch 1: train MSE 0.6263, val MSE 0.3110
epoch 2: train MSE 0.2947, val MSE 0.2523
epoch 3: train MSE 0.2752, val MSE 0.2429
epoch 4: train MSE 0.2703, val MSE 0.2379
epoch 5: train MSE 0.2626, val MSE 0.2331
epoch 6: train MSE 0.2604, val MSE 0.2295
